# ETL e persistência em nuvem do MVP

**Objetivo deste notebook:** demonstrar coleta, transformação, gravação e leitura de dados persistidos no **Google Drive**, usando o conjunto oficial AI4I 2020 da UCI (id 601).

- **Bronze:** todas as colunas originais recebidas pelo `ucimlrepo`, sem excluir registros ou atributos; o DataFrame da API é serializado para CSV.
- **Prata:** seis atributos de entrada e o alvo `Machine failure`, exatamente como no MVP de manutenção preditiva.
- **Verificação:** os dois arquivos são relidos do Drive e confrontados com os DataFrames antes da gravação.
- **Evidência:** relatório JSON com data, versões, atributos, dimensões, licença, hash dos arquivos e verificações.

**Limite de escopo:** este notebook realiza o ETL e não treina classificadores. Os arquivos são criados no Google Drive da conta autorizada.

**Fonte e licença:** [UCI AI4I 2020](https://archive.ics.uci.edu/dataset/601/ai4i), DOI 10.24432/C5HS5C, **CC BY 4.0**. Os dados são sintéticos. Esta execução não equivale a validação industrial.

In [ ]:
# Execute uma vez no início da sessão Colab.
%pip -q install ucimlrepo==0.0.7

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform

import pandas as pd
from ucimlrepo import fetch_ucirepo

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('ucimlrepo:', importlib.metadata.version('ucimlrepo'))

# Coleta oficial: as três matrizes são obtidas na mesma chamada.
fonte = fetch_ucirepo(id=601)
bronze = fonte.data.original.copy().reset_index(drop=True)
X = fonte.data.features.copy().reset_index(drop=True)
y = fonte.data.targets['Machine failure'].copy().reset_index(drop=True)

colunas_entradas = [
    'Type', 'Air temperature', 'Process temperature',
    'Rotational speed', 'Torque', 'Tool wear'
]
assert list(X.columns) == colunas_entradas, f'Entradas inesperadas: {list(X.columns)}'
assert len(bronze) == len(X) == len(y) == 10000, 'Volume inesperado'
assert int((y == 1).sum()) == 339, 'Distribuição de falhas inesperada'

# Transformação mínima: somente entradas de papel Feature e o alvo selecionado.
# Os demais alvos (TWF/HDF/PWF/OSF/RNF) não entram na tabela de modelagem.
prata = pd.concat([X, y.rename('Machine failure')], axis=1)
assert prata.shape == (10000, 7)
assert prata.isna().sum().sum() == 0
assert set(prata['Type'].unique()) <= {'L', 'M', 'H'}
assert set(prata['Machine failure'].unique()) <= {0, 1}

print('BRONZE:', bronze.shape, '| PRATA:', prata.shape)
print('Classes:', prata['Machine failure'].value_counts().to_dict())
print('Atributos prata:', list(prata.columns))

## Autorizar o Google Drive

A célula abaixo pede acesso ao **Google Drive da conta escolhida**. Confira a solicitação de permissão antes de aceitar. O notebook só grava os arquivos do projeto na pasta indicada; ele não lê outros documentos da conta.

**Local padrão:** `Meu Drive/UnB/SSD/MVP_Manutencao_Preditiva/execucoes/<data-hora-UTC>/`.

Cada execução usa uma nova subpasta para **não sobrescrever** arquivos anteriores.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

pasta_base = Path('/content/drive/MyDrive/UnB/SSD/MVP_Manutencao_Preditiva/execucoes')
identificador_execucao = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
pasta_execucao = pasta_base / identificador_execucao
pasta_bronze = pasta_execucao / 'bronze'
pasta_prata = pasta_execucao / 'prata'
pasta_evidencias = pasta_execucao / 'evidencias'

pasta_execucao.mkdir(parents=True, exist_ok=False)
for pasta in (pasta_bronze, pasta_prata, pasta_evidencias):
    pasta.mkdir(parents=True, exist_ok=True)

print('Destino (Drive):', pasta_execucao)

In [ ]:
# Carga (persistência): CSVs gravados no diretório montado do Drive.
arquivo_bronze = pasta_bronze / 'ai4i_original_ucimlrepo.csv'
arquivo_prata = pasta_prata / 'ai4i_modelagem.csv'

bronze.to_csv(arquivo_bronze, index=False, encoding='utf-8', float_format='%.17g')
prata.to_csv(arquivo_prata, index=False, encoding='utf-8', float_format='%.17g')

print('Arquivos gravados em nuvem:')
print(' -', arquivo_bronze, '| bytes:', arquivo_bronze.stat().st_size)
print(' -', arquivo_prata, '| bytes:', arquivo_prata.stat().st_size)

In [ ]:
# Verificação de persistência: reler arquivos DIRETAMENTE do Drive.
bronze_relida = pd.read_csv(arquivo_bronze)
prata_relida = pd.read_csv(arquivo_prata)

# Não exigir identidade binária entre representações em memória e CSV.
# Exigir a mesma estrutura, categorias, inteiros e valores numéricos (tolerância de serialização).
pd.testing.assert_frame_equal(bronze, bronze_relida, check_dtype=False,
                              check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(prata, prata_relida, check_dtype=False,
                              check_exact=False, rtol=1e-12, atol=1e-12)
assert prata_relida.shape == (10000, 7)
assert int(prata_relida['Machine failure'].sum()) == 339
assert list(prata_relida.columns) == colunas_entradas + ['Machine failure']

def sha256_arquivo(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(bloco)
    return digest.hexdigest()

executado_em = datetime.now(timezone.utc).isoformat()
relatorio = {
    'projeto': 'MVP SSD — Manutenção Preditiva',
    'etapa': 'Coleta, transformação, persistência e verificação da carga na nuvem',
    'data_verificacao_utc': executado_em,
    'fonte': {
        'nome': 'AI4I 2020 Predictive Maintenance Dataset',
        'repositorio': 'UCI Machine Learning Repository',
        'id_uci': 601,
        'doi': '10.24432/C5HS5C',
        'licenca_dados': 'CC BY 4.0',
        'metodo': 'fetch_ucirepo(id=601)',
        'formato_persistido': 'CSV UTF-8',
        'observacao_bronze': 'DataFrame original retornado pelo ucimlrepo, serializado em CSV; não é copia byte a byte do arquivo original da UCI.'
    },
    'ambiente': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'ucimlrepo': importlib.metadata.version('ucimlrepo'),
    },
    'transformacoes': [
        'Selecao das seis colunas com papel Feature, sem excluir linhas',
        'Selecao apenas do alvo Machine failure, excluindo outros alvos da modelagem',
        'Concatenacao por ordem dos registros, preservando 10000 linhas',
        'Serializacao dos DataFrames em CSV UTF-8'
    ],
    'arquivos_no_drive': {
        'bronze': {
            'caminho_relativo_execucao': 'bronze/ai4i_original_ucimlrepo.csv',
            'linhas': int(bronze_relida.shape[0]),
            'colunas': int(bronze_relida.shape[1]),
            'bytes': arquivo_bronze.stat().st_size,
            'sha256': sha256_arquivo(arquivo_bronze),
        },
        'prata': {
            'caminho_relativo_execucao': 'prata/ai4i_modelagem.csv',
            'linhas': int(prata_relida.shape[0]),
            'colunas': list(prata_relida.columns),
            'falhas': int(prata_relida['Machine failure'].sum()),
            'bytes': arquivo_prata.stat().st_size,
            'sha256': sha256_arquivo(arquivo_prata),
        },
    },
    'validacoes': {
        'arquivos_reabertos_do_drive': True,
        'conteudo_compativel_com_original_apos_releitura': True,
        'linhas_esperadas': 10000,
        'falhas_esperadas': 339,
        'aprovado': True,
    },
}

arquivo_relatorio = pasta_evidencias / 'relatorio_etl_nuvem.json'
arquivo_relatorio.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
# Confirma que o relatório também foi persistido e pode ser lido.
verificacao_relatorio = json.loads(arquivo_relatorio.read_text(encoding='utf-8'))
assert verificacao_relatorio['validacoes']['aprovado'] is True

print('\nETL E PERSISTÊNCIA EM NUVEM: VERIFICADOS')
print('Relatório:', arquivo_relatorio)
print('Bronze:', bronze_relida.shape)
print('Prata:', prata_relida.shape)
print('SHA-256 bronze:', relatorio['arquivos_no_drive']['bronze']['sha256'])
print('SHA-256 prata:', relatorio['arquivos_no_drive']['prata']['sha256'])

## Conferência da execução

1. Confira se apareceu **`ETL E PERSISTÊNCIA EM NUVEM: VERIFICADOS`**.
2. No Google Drive, abra a pasta `UnB/SSD/MVP_Manutencao_Preditiva/execucoes/` e faça uma captura de tela mostrando as pastas `bronze`, `prata` e `evidencias` da execução.
3. Verifique o arquivo persistido `evidencias/relatorio_etl_nuvem.json`; ele registra as dimensões e os hashes dos CSVs.

**Importante:** o código e o JSON comprovam uma execução concluída e arquivos acessíveis via Drive montado. Uma captura da pasta no site do Google Drive fornece evidência visual adicional de persistência. Isso não equivale à implantação de um serviço de Machine Learning.